# Character and Entity Networks

**Computational Text Analysis Pack · Module 19 — Networks from Text**

*Technique:* Co-occurrence networks of people; Moretti's plot networks and their critics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/19b_character_networks.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Moretti's "Network Theory, Plot Analysis" proposed treating a play or novel as a
network of characters, connected when they appear together. It is one of the
most cited and most contested moves in computational literary studies.

We build one, using the rule-based entity extraction from module 15c, and then
take the criticisms seriously — because this method makes very pretty figures
out of very thin evidence, and knowing where the thinness is matters more than
the figure.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import networkx as nx
import re

work = (df_colonial.groupby(["work_id", "title", "author"])["text"]
        .apply(" ".join).reset_index())
work["n"] = work["text"].str.split().str.len()
chosen = work.sort_values("n", ascending=False).iloc[0]
print(f"'{chosen['title'][:60]}' by {chosen['author']} ({chosen['n']:,} words)")

TITLES = r"(?:Mr|Mrs|Miss|Dr|Sir|Lord|Lady|Captain|Colonel|Chief|King|Queen)"
names = Counter(re.findall(rf"\b{TITLES}\.?\s+([A-Z][a-z]+)", chosen["text"]))
names += Counter(w for w in re.findall(r"\b[A-Z][a-z]{3,}\b", chosen["text"]))
characters = [n for n, c in names.most_common(60) if c >= 8][:22]
print(f"\n{len(characters)} candidate characters: {', '.join(characters[:12])} ...")

## Co-occurrence in a window

Two characters are linked if they appear within the same span of text. The span
is a modelling choice with no natural value — paragraph, page, chapter — and it
changes the graph completely.

In [ ]:
toks = chosen["text"].split()
SPAN = 220

windows = [set(w.strip(".,;:!?\"'") for w in toks[i:i + SPAN])
           for i in range(0, len(toks) - SPAN, SPAN // 2)]
co = Counter()
solo = Counter()
for win in windows:
    present = [c for c in characters if c in win]
    for c in present:
        solo[c] += 1
    for i in range(len(present)):
        for j in range(i + 1, len(present)):
            co[tuple(sorted((present[i], present[j])))] += 1

G = nx.Graph()
for c in characters:
    if solo[c]:
        G.add_node(c, appearances=solo[c])
for (a, b), n in co.items():
    if n >= 2:
        G.add_edge(a, b, weight=n)
G.remove_nodes_from([n for n in list(G) if G.degree(n) == 0])
print(f"{G.number_of_nodes()} characters, {G.number_of_edges()} links "
      f"(window of {SPAN} words)")

In [ ]:
plt.figure(figsize=(9.5, 7))
pos = nx.spring_layout(G, k=0.6, seed=3)
sizes = [80 + 12 * G.nodes[n]["appearances"] for n in G]
widths = [0.4 + 0.35 * G[u][v]["weight"] for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=widths, alpha=.3)
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color="steelblue", alpha=.85)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title(f"Character network — {chosen['title'][:45]}")
plt.axis("off")
plt.tight_layout()

In [ ]:
print(f"{'character':<16}{'appearances':>13}{'degree':>9}{'betweenness':>14}")
print("-" * 52)
btw = nx.betweenness_centrality(G)
for n in sorted(G, key=lambda n: -G.nodes[n]["appearances"])[:10]:
    print(f"{n:<16}{G.nodes[n]['appearances']:>13}{G.degree(n):>9}{btw[n]:>14.3f}")

## What the critics say, and why they are right to

**Bode's critique** (2017) argues that Moretti and Jockers model literary systems
ahistorically, neglecting the textual scholarship that determines what the text
even *is*. That objection lands hard here, and you can see it in our own
pipeline:

- Our "characters" came from a **capitalisation regex**. Place names, months and
  sentence-initial words are in that list. We never disambiguated.
- Co-occurrence in a 220-word window is not interaction. Two characters
  mentioned in the same paragraph may never meet.
- The edition matters. This text came from Project Gutenberg with headers
  stripped by a script; a different edition gives a different network.

**Data Feminism** (D'Ignazio and Klein) adds the question of who is *missing*:
a network built from named, capitalised entities systematically over-represents
whoever the text bothered to name.

In [ ]:
# Test the last point directly: does the window size change the story?
print(f"{'window':>8}{'nodes':>8}{'edges':>8}{'density':>10}{'top by betweenness':>24}")
print("-" * 58)
for span in [80, 150, 220, 400, 800]:
    wins = [set(w.strip(".,;:!?\"'") for w in toks[i:i + span])
            for i in range(0, len(toks) - span, span // 2)]
    c2 = Counter()
    for win in wins:
        pres = [c for c in characters if c in win]
        for i in range(len(pres)):
            for j in range(i + 1, len(pres)):
                c2[tuple(sorted((pres[i], pres[j])))] += 1
    g = nx.Graph([(a, b) for (a, b), n in c2.items() if n >= 2])
    if g.number_of_nodes() < 2:
        continue
    b2 = nx.betweenness_centrality(g)
    top = max(b2, key=b2.get)
    print(f"{span:>8}{g.number_of_nodes():>8}{g.number_of_edges():>8}"
          f"{nx.density(g):>10.3f}{top:>24}")
print()
print("If the most 'central' character changes with the window, centrality was")
print("a property of the parameter, not of the plot. Report the window, and")
print("check this table before making a claim about a protagonist.")

### 🔧 Your turn 1

Prune the character list by hand — remove place names and months — and rebuild. How much does the network change?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually a great deal, and often the most 'central' node was a place or a
frequently capitalised common noun. That is the practical form of Bode's
objection: the interesting-looking structure was partly an artefact of a lazy
extraction step. The honest workflow is manual verification of the entity list
before any network claim — which is slow, unglamorous, and the difference
between a figure and an argument.

</details>